# Bundle adjustment: structure from motion by Levenberg–Marquardt

*graphics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/bundle_adjustment)

Several cameras photograph the same 3D points, with pixel noise. Starting from rough poses (axis-angle rotations $\boldsymbol\omega_c$, translations $\mathbf t_c$) and rough points $\mathbf X_p$, bundle adjustment minimises the total reprojection error $\tfrac12\sum_{c,p}\lVert\pi(R_c\mathbf X_p + \mathbf t_c) - \tilde{\mathbf u}_{cp}\rVert^2$ by Levenberg–Marquardt. The Jacobian comes from autograd, the normal equations $J^{\mathsf T}J\,\Delta = -J^{\mathsf T}\mathbf r$ are solved through the Schur complement, and fixing the 7 gauge freedoms makes the solution unique.

**Vector form**

$$E(\boldsymbol\theta) = \tfrac12\sum_{c,p}\big\lVert \pi\big(R(\boldsymbol\omega_c)\,\mathbf X_p + \mathbf t_c\big) - \tilde{\mathbf u}_{cp}\big\rVert^2,\qquad \big(J^{\mathsf T}J + \lambda\,\mathrm{diag}\,J^{\mathsf T}J\big)\,\Delta = -J^{\mathsf T}\mathbf r,\qquad \boldsymbol\theta \leftarrow \boldsymbol\theta + \Delta$$

**Index (tensor) form**

$$x_{cpi} = R_{cij}X_{pj} + t_{ci},\qquad r_{cpa} = f\,\frac{x_{cpa}}{x_{cp3}} + c_a - \tilde u_{cpa},\qquad J_{mn} = \frac{\partial r_m}{\partial\theta_n},\qquad J_{mn}J_{mn'}\,\Delta_{n'} = -J_{mn}\,r_m$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\omega_{ci},\ t_{ci}$ | `true_cameras(n_cams)` | (C, 3), (C, 3) | camera pose: axis-angle rotation (direction = axis, length = angle) and translation, so $x_{\rm cam} = R\mathbf X + \mathbf t$ |
| $R_{cij}$ | `rodrigues(omega)` | (C, 3, 3) | Rodrigues formula $I + \sin\theta\,K + (1-\cos\theta)K^2$; three numbers per rotation, no constraints to enforce |
| $x_{cpi} = R_{cij}X_{pj} + t_{ci}$ | `einsum('cij,pj->cpi', R, X) + t[:, None]` | (C, P, 3) | every point in every camera frame, $j$ summed |
| $r_{cpa}$ | `project(x_cam, f, c) - observed` | (C, P, 2) | reprojection error in pixels; flattened to $r_m$, $m = (c, p, a)$ |
| $\theta_n$ | `full_start[free]` | (n,) | free unknowns: all poses and points minus the 7 gauge numbers (camera 0 and $t_x$ of camera 1) |
| $J_{mn}$ | `torch.func.jacrev(fn)(theta)` | (2CP, n) | Jacobian by reverse-mode autograd, one vectorised backward pass per row |
| $J_{mn}J_{mn'},\ J_{mn}r_m$ | `einsum('mn,mk->nk', J, J), einsum('mn,m->n', J, r)` | (n, n), (n,) | Gauss–Newton matrix $H = J^{\mathsf T}J$ and gradient $J^{\mathsf T}\mathbf r$, $m$ summed |
| $H = \begin{pmatrix} U & W \\ W^{\mathsf T} & V\end{pmatrix}$ | `H[:n_cam, :n_cam], H[:n_cam, n_cam:], …` | (n_c, n_c), (n_c, 3P) | cameras / points blocks; a residual touches one camera and one point, so $V$ is block diagonal |
| $S = U - W V^{-1} W^{\mathsf T}$ | `U - einsum('apj,pjk,bpk->ab', W, V_inv, W)` | (n_c, n_c) | Schur complement (reduced camera system): $P$ tiny $3\times 3$ inverses, then one small solve |
| $\Delta_p = V_p^{-1}(-g_p - W_p^{\mathsf T}\Delta_c)$ | `einsum('pjk,pk->pj', V_inv, ...)` | (P, 3) | back-substitution: each point is updated on its own |
| $\lambda$ | `lam` | () | Levenberg–Marquardt damping: divided by 3 after a successful step, multiplied by 4 after a failed one |

The data are synthetic: points in a ball of radius 1, cameras on an arc 5 units away (focal length 1000 px, a $640\times 640$ image), every point seen by every camera, Gaussian pixel noise $\sigma$. The start perturbs every rotation by a few degrees and every translation and point by a fraction of the scene size, so the first reprojection errors are tens of pixels. Gauge freedom: moving, rotating or scaling the whole scene together with all the cameras changes no image, so $E$ has a 7-dimensional valley of equal minima and $J^{\mathsf T}J$ has 7 zero eigenvalues (the console prints them, about $10^{-16}$ of the largest). Fixing camera 0 (6 numbers) and the $x$-translation of camera 1 (the scale) removes the null space: $J^{\mathsf T}J$ becomes full rank and the steps are well defined. We fix them at their true values, so the result can be compared with the truth directly. Each iteration linearises the residuals, $\mathbf r(\boldsymbol\theta + \Delta) \approx \mathbf r + J\Delta$, and solves the damped normal equations; the Jacobian is built by torch.func.jacrev, i.e. autograd vectorised over all residuals, which removes the error-prone hand derivation. The Schur complement exploits the sparsity (the arrow-shaped $J^{\mathsf T}J$ panel): eliminating the points first leaves a system the size of the cameras only, which is why real bundle adjusters handle millions of points. Here the dense and Schur solves give the same step to about $10^{-12}$. At the optimum the RMS reprojection error equals the noise floor $\sigma\sqrt{1 - n/m}$ (each fitted unknown absorbs a little of the noise) and the residual histogram matches the noise distribution: there is nothing left to explain. The remaining point and camera errors are set by the noise and the geometry, not by the optimiser. Reference: B. Triggs, P. McLauchlan, R. Hartley and A. Fitzgibbon, "Bundle Adjustment — A Modern Synthesis" (Vision Algorithms: Theory and Practice, 1999). Simplifications: no outliers (real systems use a robust loss), no lens distortion, all points visible in all cameras, dense Jacobian storage. Try a larger start perturbation (LM then needs more damping, or falls into a wrong minimum), more noise, or only 2 cameras.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/bundle_adjustment/main.py
"""Main loop: true scene → noisy image points → perturbed start → gauge fixing → Levenberg–Marquardt → compare."""

import math
import time
from pathlib import Path

import torch

from problems._common.runtime import cli, fast_mode

from . import plot, solve, tensors
from .equation import normal_equations, project, rms, rodrigues, to_camera

SEED = 0
SHOW_CAM = 2  # the camera whose image is drawn


def run(params: dict, out_dir: Path) -> Path:
    fast = fast_mode()
    n_cams = params["n_cameras"]
    n_pts = 30 if fast else params["n_points"]
    iterations = 6 if fast else params["iterations"]
    sigma = params["noise"]
    f, c = tensors.FOCAL, tensors.IMAGE / 2
    # float64 on the CPU: the problem is small, and Gauss–Newton needs the precision near the optimum.

    X_true = tensors.true_points(n_pts, SEED)  # (P, 3)
    omega_true, t_true = tensors.true_cameras(n_cams)  # (C, 3), (C, 3)
    full_true = tensors.pack(omega_true, t_true, X_true)
    gen = torch.Generator().manual_seed(SEED + 2)
    clean = project(to_camera(omega_true, t_true, X_true), f, c)  # (C, P, 2)
    observed = clean + sigma * torch.randn(clean.shape, generator=gen, dtype=clean.dtype)

    free = tensors.gauge_mask(n_cams, n_pts)
    n_free, n_cam = int(free.sum()), 6 * n_cams - 7
    m = observed.numel()
    full_start = tensors.perturb(
        full_true, n_cams, params["init_rot"], params["init_noise"], params["init_noise"], SEED
    )
    full_start[~free] = full_true[~free]  # the gauge: camera 0 and camera 1's x-translation at their true values
    fn = solve.residual_function(full_start, free, n_cams, observed, f, c)
    print(f"{n_cams} cameras (f = {f:.0f} px) see {n_pts} points: {m} residuals, {6 * n_cams + 3 * n_pts} "
          f"unknowns − 7 gauge = {n_free} free ({n_cam} camera + {3 * n_pts} point)")  # fmt: skip

    # the gauge freedom: without fixing, JᵀJ has a 7-dimensional null space (rotation, translation, scale)
    fn_all = solve.residual_function(full_start, torch.ones_like(free), n_cams, observed, f, c)
    H_all, _ = normal_equations(solve.jacobian(fn_all, full_start), fn_all(full_start))
    ev = torch.linalg.eigvalsh(H_all)
    print(f"all unknowns free: rank JᵀJ = {int(torch.linalg.matrix_rank(H_all))} of {len(ev)}; smallest 8 "
          f"eigenvalues / largest: " + " ".join(f"{e:.0e}" for e in (ev[:8] / ev[-1]).tolist()))  # fmt: skip

    theta0 = full_start[free].clone()
    t0 = time.perf_counter()
    theta, hist, H = solve.levenberg_marquardt(fn, theta0, n_cam, iterations, params["solver"])
    t_lm = time.perf_counter() - t0
    r0, r1 = fn(theta0), fn(theta)
    floor = sigma * math.sqrt((m - n_free) / m)  # expected RMS at the optimum: noise minus fitted degrees of freedom
    print(f"gauge fixed: rank JᵀJ = {int(torch.linalg.matrix_rank(H))} of {n_free} (full rank)")
    print(f"Levenberg–Marquardt ({params['solver']} solve), {iterations} iterations in {t_lm:.2f} s: RMS "
          f"reprojection error {rms(r0):.2f} → {rms(r1):.3f} px (noise floor σ√(1−n/m) = {floor:.3f} px)")  # fmt: skip

    # the two linear solvers agree
    g = torch.einsum("mn,m->n", solve.jacobian(fn, theta0), fn(theta0))
    H0, _ = normal_equations(solve.jacobian(fn, theta0), fn(theta0))
    d_dense, d_schur = solve.dense_step(H0, g), solve.schur_step(H0, g, n_cam)
    print(f"Schur complement: reduced camera system {n_cam}×{n_cam} instead of {n_free}×{n_free}; "
          f"max |Δ_schur − Δ_dense| = {(d_dense - d_schur).abs().max():.1e}")  # fmt: skip

    errs = [_errors(fn, th, free, full_start, full_true, n_cams) for th in hist["theta"]]
    print(f"point error {errs[0][0]:.3f} → {errs[-1][0]:.4f}   ·   camera-centre error {errs[0][1]:.3f} → "
          f"{errs[-1][1]:.4f}  (scene radius 1, cameras {tensors.RADIUS:.0f} away)")  # fmt: skip

    frames = []
    for th in hist["theta"]:
        full = full_start.clone()
        full[free] = th
        omega, t, X = tensors.unpack(full, n_cams)
        frames.append({"X": X, "centre": _centres(omega, t), "R": rodrigues(omega),
                       "proj": project(to_camera(omega, t, X), f, c)[SHOW_CAM], "r": fn(th)})  # fmt: skip
    truth = {"X": X_true, "centre": _centres(omega_true, t_true), "R": rodrigues(omega_true)}
    return plot.animate(frames, truth, observed[SHOW_CAM], hist["cost"], errs, H, n_cam, sigma, floor,
                        SHOW_CAM, out_dir / "bundle_adjustment.gif")  # fmt: skip


def _centres(omega: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Camera centres in the world: C = -Rᵀ t.   (C, 3)."""
    return -torch.einsum("cji,cj->ci", rodrigues(omega), t)


def _errors(fn, theta, free, full_start, full_true, n_cams) -> tuple[float, float]:
    """RMS distance of the points and of the camera centres from the truth (same gauge, so no alignment needed)."""
    full = full_start.clone()
    full[free] = theta
    omega, t, X = tensors.unpack(full, n_cams)
    o_true, t_true, X_true = tensors.unpack(full_true, n_cams)
    e_pts = (X - X_true).norm(dim=1).pow(2).mean().sqrt().item()
    e_cam = (_centres(omega, t) - _centres(o_true, t_true)).norm(dim=1).pow(2).mean().sqrt().item()
    return e_pts, e_cam


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/bundle_adjustment/equation.py
r"""Physics → tensor translation.

Bundle adjustment finds the camera poses and 3D points that best explain what the cameras saw. Camera c has a
rotation R_c = exp([ω_c]×) (axis-angle ω_c) and a translation t_c; point p has position X_p. Indices: c = camera,
p = point, i, j = 3D axes, a = image axis (u, v), m = residual, n = unknown.

    rotation      R_{cij} = δ_{ij} + sin θ_c K_{cij} + (1 - cos θ_c) K_{cik} K_{ckj}   (Rodrigues, K = [ω_c/θ_c]×)
    to camera     x_{cpi} = R_{cij} X_{pj} + t_{ci}                     j summed
    projection    u_{cpa} = f x_{cpa} / x_{cp3} + c_a
    residual      r_{cpa} = u_{cpa} - ũ_{cpa}                           ũ = the measured (noisy) image point
    cost          E = ½ r_m r_m                                          m = (c, p, a) flattened, summed

Gauss–Newton linearises r(θ + Δ) ≈ r + J Δ with the Jacobian J_{mn} = ∂r_m / ∂θ_n (from autograd) and solves the
normal equations (JᵀJ)_{nn'} Δ_{n'} = -(Jᵀr)_n; Levenberg–Marquardt adds λ diag(JᵀJ) to keep each step safe.
"""

import torch


def rodrigues(omega: torch.Tensor) -> torch.Tensor:
    """Axis-angle ω (C, 3) → rotation matrices R (C, 3, 3) = I + sin θ K + (1 - cos θ) K², θ = |ω|, K = [ω/θ]×."""
    theta = torch.sqrt((omega**2).sum(-1) + 1e-20)  # (C,); the tiny shift keeps the gradient finite at ω = 0
    k = omega / theta[:, None]
    zero = torch.zeros_like(theta)
    K = torch.stack(
        [
            torch.stack([zero, -k[:, 2], k[:, 1]], -1),
            torch.stack([k[:, 2], zero, -k[:, 0]], -1),
            torch.stack([-k[:, 1], k[:, 0], zero], -1),
        ],
        -2,
    )  # (C, 3, 3) cross-product matrix: K v = k × v
    eye = torch.eye(3, dtype=omega.dtype, device=omega.device)
    s, c = torch.sin(theta)[:, None, None], torch.cos(theta)[:, None, None]
    return eye + s * K + (1 - c) * torch.einsum("cik,ckj->cij", K, K)


def to_camera(omega: torch.Tensor, t: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    """x_{cpi} = R_{cij} X_{pj} + t_{ci}   (j summed): every point in every camera frame.   → (C, P, 3)."""
    return torch.einsum("cij,pj->cpi", rodrigues(omega), X) + t[:, None, :]


def project(x_cam: torch.Tensor, focal: float, centre: float) -> torch.Tensor:
    """Pinhole camera u_a = f x_a / x_3 + c_a.   (..., 3) → (..., 2) pixels."""
    return focal * x_cam[..., :2] / x_cam[..., 2:3] + centre


def residuals(omega, t, X, observed, focal, centre) -> torch.Tensor:
    """Reprojection errors r_{cpa} = π(R_c X_p + t_c) - ũ_{cp}: (C, P, 2) pixels."""
    return project(to_camera(omega, t, X), focal, centre) - observed


def cost(r: torch.Tensor) -> torch.Tensor:
    """E = ½ Σ_m r_m²: the total squared reprojection error."""
    return 0.5 * (r**2).sum()


def rms(r: torch.Tensor) -> torch.Tensor:
    """Root-mean-square reprojection error per image coordinate, in pixels."""
    return torch.sqrt((r**2).mean())


def normal_equations(J: torch.Tensor, r: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """H_{nn'} = J_{mn} J_{mn'} and g_n = J_{mn} r_m   (m summed): the Gauss–Newton matrix JᵀJ and gradient Jᵀr."""
    return torch.einsum("mn,mk->nk", J, J), torch.einsum("mn,m->n", J, r)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/bundle_adjustment/tensors.py
"""Build the concrete tensors: true 3D points and cameras, noisy observations, the perturbed start and the gauge."""

import math

import torch

FOCAL = 1000.0  # px
IMAGE = 640  # px, square image; principal point at the centre
RADIUS = 5.0  # camera distance from the origin


def true_points(n_points: int, seed: int) -> torch.Tensor:
    """X_{pj}: n_points random points inside a ball of radius 1 around the origin.   → (P, 3) float64."""
    gen = torch.Generator().manual_seed(seed)
    x = torch.randn(n_points, 3, generator=gen, dtype=torch.float64)
    r = torch.rand(n_points, 1, generator=gen, dtype=torch.float64) ** (1 / 3)
    return x / x.norm(dim=1, keepdim=True) * r


def rotmat_to_axis_angle(R: torch.Tensor) -> torch.Tensor:
    """Log map of SO(3), valid for rotation angles below π: (C, 3, 3) → ω (C, 3) with |ω| = angle."""
    cos = ((R.diagonal(dim1=-2, dim2=-1).sum(-1) - 1) / 2).clamp(-1, 1)
    angle = torch.acos(cos)
    v = torch.stack([R[:, 2, 1] - R[:, 1, 2], R[:, 0, 2] - R[:, 2, 0], R[:, 1, 0] - R[:, 0, 1]], dim=-1)
    scale = torch.where(angle > 1e-8, angle / (2 * torch.sin(angle).clamp(min=1e-12)), 0.5)
    return v * scale[:, None]


def true_cameras(n_cams: int) -> tuple[torch.Tensor, torch.Tensor]:
    """n_cams cameras on an arc in front of the points (azimuth -50° … 50°, alternately 10° above/below), all looking
    at the origin. Returns axis-angle ω (C, 3) and translation t (C, 3) with x_cam = R(ω) X + t.

    Camera axes: x right, y down, z forward (depth), so camera 0 at azimuth -50° sees the points along +z rotated.
    """
    az = torch.linspace(-math.radians(50), math.radians(50), n_cams, dtype=torch.float64)
    el = torch.tensor([math.radians(10.0 if k % 2 == 0 else -10.0) for k in range(n_cams)], dtype=torch.float64)
    centre = RADIUS * torch.stack([az.sin() * el.cos(), el.sin(), -az.cos() * el.cos()], dim=-1)  # (C, 3)
    forward = -centre / centre.norm(dim=-1, keepdim=True)
    world_down = torch.tensor([0.0, 1.0, 0.0], dtype=torch.float64).expand_as(forward)
    right = torch.linalg.cross(world_down, forward)
    right = right / right.norm(dim=-1, keepdim=True)
    down = torch.linalg.cross(forward, right)
    R = torch.stack([right, down, forward], dim=1)  # (C, 3, 3) rows = camera axes in world coordinates
    t = -torch.einsum("cij,cj->ci", R, centre)
    return rotmat_to_axis_angle(R), t


def pack(omega: torch.Tensor, t: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    """All unknowns in one vector, cameras first: [ω_0, t_0, ω_1, t_1, …, X_0, X_1, …]   → (6C + 3P,)."""
    return torch.cat([torch.cat([omega, t], dim=1).reshape(-1), X.reshape(-1)])


def unpack(full: torch.Tensor, n_cams: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cam = full[: 6 * n_cams].reshape(n_cams, 6)
    return cam[:, :3], cam[:, 3:], full[6 * n_cams :].reshape(-1, 3)


def gauge_mask(n_cams: int, n_points: int) -> torch.Tensor:
    """True for the free unknowns. Fixed: all 6 numbers of camera 0 (rotation + position of the world frame) and the
    x-translation of camera 1 (the overall scale): 7 gauge freedoms, the similarity group of 3D space."""
    free = torch.ones(6 * n_cams + 3 * n_points, dtype=torch.bool)
    free[:6] = False  # camera 0
    free[6 + 3] = False  # t_x of camera 1
    return free


def perturb(full: torch.Tensor, n_cams: int, rot_deg: float, trans: float, point: float, seed: int) -> torch.Tensor:
    """A rough start: random rotations of ~rot_deg degrees, translation and point noise (world units)."""
    gen = torch.Generator().manual_seed(seed + 1)
    omega, t, X = unpack(full.clone(), n_cams)
    omega = omega + math.radians(rot_deg) * torch.randn(omega.shape, generator=gen, dtype=full.dtype)
    t = t + trans * torch.randn(t.shape, generator=gen, dtype=full.dtype)
    X = X + point * torch.randn(X.shape, generator=gen, dtype=full.dtype)
    return pack(omega, t, X)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/bundle_adjustment/solve.py
"""Levenberg–Marquardt on the reprojection error, with the Jacobian from autograd (torch.func.jacrev) and the
normal equations solved either densely or by the Schur complement (eliminating the points first)."""

import torch

from problems._common.progress import steps

from .equation import cost, normal_equations, residuals
from .tensors import unpack


def residual_function(full_fixed, free, n_cams, observed, focal, centre):
    """r(θ): the flat residual vector (2CP,) as a function of the free unknowns θ only.

    The gauge-fixed numbers stay at their values in full_fixed; E is the 0/1 matrix that places θ into the full
    vector, full = full_fixed (free entries zeroed) + E θ.
    """
    E = torch.eye(len(free), dtype=full_fixed.dtype, device=full_fixed.device)[:, free]  # (N_full, n_free)
    base = torch.where(free.to(full_fixed.device), 0.0, full_fixed)

    def fn(theta: torch.Tensor) -> torch.Tensor:
        full = base + torch.einsum("Nn,n->N", E, theta)
        omega, t, X = unpack(full, n_cams)
        return residuals(omega, t, X, observed, focal, centre).reshape(-1)

    return fn


def jacobian(fn, theta: torch.Tensor) -> torch.Tensor:
    """J_{mn} = ∂r_m / ∂θ_n by reverse-mode autograd, vectorised over the rows m: (2CP, n_free)."""
    return torch.func.jacrev(fn)(theta)


def dense_step(H: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """Solve H Δ = -g for all unknowns at once (cost grows like n³)."""
    return torch.linalg.solve(H, -g)


def schur_step(H: torch.Tensor, g: torch.Tensor, n_cam: int) -> torch.Tensor:
    """Solve H Δ = -g using the block structure H = [[U, W], [Wᵀ, V]] (cameras, points).

    Each residual involves one camera and ONE point, so V is block diagonal: P separate 3 × 3 blocks, inverted
    independently. Eliminating the points leaves the small reduced camera system
        S Δ_c = -g_c + W V⁻¹ g_p,   S = U - W V⁻¹ Wᵀ,
    then each point's update follows on its own: Δ_p = V_p⁻¹ (-g_p - Wᵀ Δ_c).
    """
    U, W = H[:n_cam, :n_cam], H[:n_cam, n_cam:]  # (n_c, n_c), (n_c, 3P)
    n_pts = (H.shape[0] - n_cam) // 3
    V_blocks = H[n_cam:, n_cam:].reshape(n_pts, 3, n_pts, 3).diagonal(dim1=0, dim2=2).permute(2, 0, 1)  # (P, 3, 3)
    V_inv = torch.linalg.inv(V_blocks)  # P small inverses instead of one 3P × 3P
    Wp = W.reshape(n_cam, n_pts, 3)  # (n_c, P, 3)
    g_c, g_p = g[:n_cam], g[n_cam:].reshape(n_pts, 3)
    S = U - torch.einsum("apj,pjk,bpk->ab", Wp, V_inv, Wp)  # p, j, k summed
    rhs = -g_c + torch.einsum("apj,pjk,pk->a", Wp, V_inv, g_p)
    d_c = torch.linalg.solve(S, rhs)
    d_p = torch.einsum("pjk,pk->pj", V_inv, -g_p - torch.einsum("apk,a->pk", Wp, d_c))
    return torch.cat([d_c, d_p.reshape(-1)])


def levenberg_marquardt(fn, theta: torch.Tensor, n_cam: int, iterations: int, method: str, lam: float = 1e-3):
    """Minimise ½|r(θ)|². Each iteration: J by autograd, H = JᵀJ, g = Jᵀr, try (H + λ diag H) Δ = -g; accept the
    step and shrink λ if the cost drops, otherwise grow λ and retry (at most 10 tries).

    Returns θ after every iteration (entry 0 = start), the costs, the damping used, and the last H.
    """
    history = {"theta": [theta.clone()], "cost": [cost(fn(theta)).item()], "lam": []}
    H = None
    for _ in steps(iterations, "LM iterations"):
        r = fn(theta)
        J = jacobian(fn, theta)  # (m, n)
        H, g = normal_equations(J, r)
        current = cost(r).item()
        for _try in range(10):
            A = H + lam * torch.diag(torch.diagonal(H))  # Marquardt's scaling: damp each unknown by its own curvature
            delta = schur_step(A, g, n_cam) if method == "schur" else dense_step(A, g)
            trial = cost(fn(theta + delta)).item()
            if trial < current:
                theta, lam = theta + delta, max(lam / 3, 1e-9)
                break
            lam *= 4
        history["theta"].append(theta.clone())
        history["cost"].append(cost(fn(theta)).item())
        history["lam"].append(lam)
    return theta, history, H


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/bundle_adjustment/plot.py
"""Animate Levenberg–Marquardt: the scene converging to the truth, one camera's reprojection errors, the block
structure of JᵀJ, the RMS error, the residual histogram and the distance to the truth."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import save_gif

from .tensors import IMAGE

BLUE, RED, GREY, DARK, ORANGE = "#2563eb", "#dc2626", "#9ca3af", "#374151", "#f59e0b"


def animate(frames, truth, observed, costs, errs, H, n_cam, sigma, floor, show_cam, path: Path) -> Path:
    m = frames[0]["r"].numel()
    rms = np.sqrt(2 * np.array(costs) / m)
    its = np.arange(len(frames))
    e_pts, e_cam = np.array([e[0] for e in errs]), np.array([e[1] for e in errs])
    obs = observed.numpy()

    fig = plt.figure(figsize=(9.0, 5.4))
    ax3 = fig.add_subplot(2, 3, 1, projection="3d")
    ax_i, ax_h = fig.add_subplot(2, 3, 2), fig.add_subplot(2, 3, 3)
    ax_r, ax_hist, ax_e = fig.add_subplot(2, 3, 4), fig.add_subplot(2, 3, 5), fig.add_subplot(2, 3, 6)

    # --- 3D scene: truth in grey, estimate in colour (x right, z forward, y down → plot with -y up) ---
    def xyz(p):
        p = p.numpy()
        return p[:, 0], p[:, 2], -p[:, 1]

    ax3.scatter(*xyz(truth["X"]), s=6, color=GREY, depthshade=False, label="true")
    ax3.scatter(*xyz(truth["centre"]), s=40, facecolors="none", edgecolors=DARK, marker="^", depthshade=False)
    pts = ax3.scatter(*xyz(frames[0]["X"]), s=6, color=BLUE, depthshade=False, label="estimate")
    cams = ax3.scatter(*xyz(frames[0]["centre"]), s=25, color=RED, marker="^", depthshade=False)
    axis_lines = [ax3.plot([], [], [], color=RED, lw=1)[0] for _ in range(len(truth["centre"]))]
    lim = 1.1 * truth["centre"].abs().max().item()
    ax3.set_xlim(-lim, lim)
    ax3.set_ylim(-lim, 1.5)
    ax3.set_zlim(-2, 2)
    ax3.view_init(elev=55, azim=-80)
    ax3.set_title("cameras (▲) and points", fontsize=8.5)
    ax3.tick_params(labelsize=6)
    ax3.legend(fontsize=6, loc="upper left")

    # --- one camera's image: measured points and the current reprojections ---
    ax_i.scatter(obs[:, 0], obs[:, 1], s=7, color=DARK, zorder=3, label="measured")
    proj = ax_i.scatter(*frames[0]["proj"].numpy().T, s=10, facecolors="none", edgecolors=BLUE, lw=0.8,
                        label="reprojected $\\pi(R X + t)$")  # fmt: skip
    res_lines = [ax_i.plot([], [], color=RED, lw=0.8)[0] for _ in range(len(obs))]
    ax_i.set_xlim(0, IMAGE)
    ax_i.set_ylim(IMAGE, 0)
    ax_i.set_aspect("equal")
    ax_i.set_title(f"image of camera {show_cam}: residuals (red)", fontsize=8.5)
    ax_i.tick_params(labelsize=6)
    ax_i.legend(fontsize=6, loc="lower right")

    # --- the sparsity of JᵀJ (static) ---
    Hn = H.abs().numpy()
    ax_h.imshow(np.log10(Hn / Hn.max() + 1e-12), cmap="Blues", vmin=-8, vmax=0, interpolation="nearest")
    ax_h.axhline(n_cam - 0.5, color=RED, lw=0.8)
    ax_h.axvline(n_cam - 0.5, color=RED, lw=0.8)
    ax_h.text(n_cam / 2, n_cam / 2, "U", ha="center", va="center", fontsize=10, color=RED)
    ax_h.text(n_cam + (Hn.shape[0] - n_cam) / 2, n_cam / 2, "W", ha="center", va="center", fontsize=10, color=RED)
    ax_h.text(n_cam + (Hn.shape[0] - n_cam) / 2, n_cam + (Hn.shape[0] - n_cam) / 2, "V: 3×3 blocks", ha="center",
              va="center", fontsize=7.5, color=RED)  # fmt: skip
    ax_h.set_title("$|J^TJ|$: cameras | points", fontsize=8.5)
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.grid(False)

    # --- RMS reprojection error ---
    ax_r.semilogy(its, rms, color=GREY, lw=0.8)
    (rms_line,) = ax_r.semilogy([], [], "o-", color=BLUE, ms=3, lw=1.3)
    ax_r.axhline(floor, color=ORANGE, ls="--", lw=1)
    ax_r.text(its[-1], floor * 1.15, "noise floor", ha="right", va="bottom", fontsize=7, color="#b45309")
    ax_r.set_xlim(0, its[-1])
    ax_r.set_title("RMS reprojection error (px)", fontsize=8.5)
    ax_r.set_xlabel("LM iteration", fontsize=7.5)
    ax_r.tick_params(labelsize=7)
    ax_r.minorticks_off()

    # --- residual histogram against the noise model N(0, σ²) ---
    edge = 5 * sigma
    bins = np.linspace(-edge, edge, 31)
    xs = np.linspace(-edge, edge, 200)
    ax_hist.plot(xs, np.exp(-0.5 * (xs / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi)), color=ORANGE, lw=1.3,
                 label=f"noise $N(0, {sigma:g}^2)$")  # fmt: skip
    ax_hist.set_xlim(-edge, edge)
    ax_hist.set_title("residuals $r_{cpa}$ (px), clipped at ±5σ", fontsize=8.5)
    ax_hist.tick_params(labelsize=7)
    ax_hist.legend(fontsize=6.5, loc="upper right")
    hist_bars = {"bars": None}

    # --- distance to the truth ---
    ax_e.semilogy(its, e_pts, color=GREY, lw=0.8)
    ax_e.semilogy(its, e_cam, color=GREY, lw=0.8)
    (ep_line,) = ax_e.semilogy([], [], "o-", color=BLUE, ms=3, lw=1.3, label="points")
    (ec_line,) = ax_e.semilogy([], [], "^-", color=RED, ms=3, lw=1.3, label="camera centres")
    ax_e.set_xlim(0, its[-1])
    ax_e.set_title("RMS distance to the truth", fontsize=8.5)
    ax_e.set_xlabel("LM iteration", fontsize=7.5)
    ax_e.tick_params(labelsize=7)
    ax_e.legend(fontsize=6.5)
    ax_e.minorticks_off()
    counter = fig.text(0.5, 0.01, "", ha="center", va="bottom", fontsize=9)

    def update(k: int):
        fr = frames[k]
        pts._offsets3d = xyz(fr["X"])
        cams._offsets3d = xyz(fr["centre"])
        for line, ctr, R in zip(axis_lines, fr["centre"], fr["R"]):
            seg = torch.stack([ctr, ctr + 1.2 * R[2]])  # optical axis = third row of R
            x, y, z = xyz(seg)
            line.set_data_3d(x, y, z)
        p = fr["proj"].numpy()
        proj.set_offsets(p)
        for line, a, b in zip(res_lines, obs, p):
            line.set_data([a[0], b[0]], [a[1], b[1]])
        if hist_bars["bars"] is not None:
            hist_bars["bars"].remove()
        r = np.clip(fr["r"].reshape(-1).numpy(), -edge * 0.999, edge * 0.999)
        hist_bars["bars"] = ax_hist.hist(r, bins=bins, density=True, color=BLUE, alpha=0.55)[2]
        top = max(1.25 / (sigma * np.sqrt(2 * np.pi)), max(b.get_height() for b in hist_bars["bars"]) * 1.05)
        ax_hist.set_ylim(0, top)
        rms_line.set_data(its[: k + 1], rms[: k + 1])
        ep_line.set_data(its[: k + 1], e_pts[: k + 1])
        ec_line.set_data(its[: k + 1], e_cam[: k + 1])
        counter.set_text(f"iteration {k}   ·   RMS reprojection error {rms[k]:.3f} px   ·   noise σ = {sigma:g} px")

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, list(range(len(frames))), path, fps=2)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_cameras': 6,
    'n_points': 80,
    'noise': 0.5,
    'init_rot': 6.0,
    'init_noise': 0.3,
    'iterations': 10,
    'solver': 'schur'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.bundle_adjustment")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.bundle_adjustment.main")
problem_dir = Path("problems/graphics/bundle_adjustment")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
